# Phase 6 — Validation sémantique, score de confiance et analyse d'erreurs

**Objectif** : évaluer la fiabilité des informations extraites en Phase 5, et proposer
un **score de confiance par champ** combinant plusieurs signaux :
- conformité au format attendu (regex par type de champ)
- score de confiance OCR moyen des mots concernés (retourné par Tesseract)
- présence d'un label proche (cohérence structurelle)
- stabilité de la valeur sous plusieurs variantes du prétraitement (Phase 4)

**Sorties** : score de confiance par champ extrait, matrice d'erreurs détaillée,
et analyse de la corrélation entre confiance et exactitude réelle.

> Ce notebook est **autonome** : il refait l'extraction (logique de la Phase 5)
> directement ici plutôt que de dépendre d'un fichier intermédiaire, afin de rester
> indépendant de l'ordre d'exécution des notebooks précédents.


In [ ]:
import json
import re
import platform
import os
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import pytesseract

if platform.system() == "Windows":
    for _path in [r"C:\Program Files\Tesseract-OCR\tesseract.exe",
                  r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"]:
        if os.path.exists(_path):
            pytesseract.pytesseract.tesseract_cmd = _path
            break

HERE = Path.cwd()
print("Dossier courant (cwd) :", HERE.resolve())


## 0. Détection automatique du dataset (identique aux notebooks précédents)

In [ ]:
def find_split_dir(candidates_roots, split_names):
    for root in candidates_roots:
        for name in split_names:
            candidate = root / name
            if (candidate / "images").is_dir() and (candidate / "annotations").is_dir():
                return candidate
    return None


def find_project_root(start, markers=("src", "dataset", "data"), max_levels=8):
    current = start
    for _ in range(max_levels):
        if any((current / m).is_dir() for m in markers):
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


PROJECT_ROOT = find_project_root(HERE)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Impossible de trouver la racine du projet depuis {HERE}.")
print("Racine du projet détectée :", PROJECT_ROOT.resolve())

POSSIBLE_ROOTS = [PROJECT_ROOT, PROJECT_ROOT / "dataset", PROJECT_ROOT / "data", HERE, HERE.parent]
POSSIBLE_ROOTS = [r for r in POSSIBLE_ROOTS if r.exists()]

RAW_DIR = find_split_dir(POSSIBLE_ROOTS, ["training_data", "raw"])
if RAW_DIR is None:
    raise FileNotFoundError("Dossier d'entraînement (training_data/raw) introuvable.")
print("Dossier train détecté :", RAW_DIR.resolve())


def build_manifest(split_dir):
    images = sorted((split_dir / "images").glob("*.png"))
    manifest = []
    for img in images:
        ann = split_dir / "annotations" / f"{img.stem}.json"
        if ann.exists():
            manifest.append({"image": img.name, "annotation": ann.name})
    return manifest


manifest = build_manifest(RAW_DIR)
print(f"{len(manifest)} documents FUNSD (train set complet)")


## 1. Extraction (logique de la Phase 5, redéfinie ici pour l'autonomie)

In [ ]:
def run_ocr_with_boxes(img, lang="eng"):
    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    text = pytesseract.image_to_string(rgb, lang=lang).strip()
    data = pytesseract.image_to_data(rgb, lang=lang, output_type=pytesseract.Output.DICT)
    words = []
    for i in range(len(data["text"])):
        w = data["text"][i].strip()
        if not w:
            continue
        conf = float(data["conf"][i]) if str(data["conf"][i]) not in ("-1",) else -1.0
        x, y, bw, bh = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
        words.append({"text": w, "box": [x, y, x + bw, y + bh], "conf": conf})
    return {"text": text, "words": words}


TARGET_FIELDS = {
    "DATE": re.compile(r"^date\b", re.IGNORECASE),
    "TO": re.compile(r"^to\s*:?$", re.IGNORECASE),
    "FROM": re.compile(r"^from\s*:?$", re.IGNORECASE),
    "CC": re.compile(r"^cc\s*:?$", re.IGNORECASE),
    "SUBJECT": re.compile(r"^subject\b", re.IGNORECASE),
    "APPROVED_BY": re.compile(r"^approved\s*by\b", re.IGNORECASE),
    "DIVISION_NAME": re.compile(r"^division\s*name\b", re.IGNORECASE),
}


def ground_truth_pairs(annotation):
    by_id = {item["id"]: item for item in annotation["form"]}
    pairs = {}
    for item in annotation["form"]:
        if item["label"] != "question":
            continue
        field_name = None
        for name, pattern in TARGET_FIELDS.items():
            if pattern.search(item["text"].strip()):
                field_name = name
                break
        if field_name is None:
            continue
        for (src, dst) in item.get("linking", []):
            target = by_id.get(dst)
            if target and target["label"] == "answer" and target["text"].strip():
                pairs[field_name] = target["text"].strip()
    return pairs


LABEL_KEYWORDS = {
    "DATE": "date", "TO": "to", "FROM": "from", "CC": "cc",
    "SUBJECT": "subject", "APPROVED_BY": "approved", "DIVISION_NAME": "division",
}


def _distance(b1, b2):
    c1 = ((b1[0] + b1[2]) / 2, (b1[1] + b1[3]) / 2)
    c2 = ((b2[0] + b2[2]) / 2, (b2[1] + b2[3]) / 2)
    return ((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2) ** 0.5


def extract_layout_based(ocr_words, max_value_words=4):
    """Retourne aussi les mots (avec conf OCR) utilisés pour chaque valeur, et si
    un label a été trouvé à proximité (nécessaire pour le score de confiance)."""
    results = {}
    used = set()
    for i, w in enumerate(ocr_words):
        clean = w["text"].rstrip(":").strip().lower()
        matched_field = None
        for field, kw in LABEL_KEYWORDS.items():
            if clean == kw or clean.startswith(kw):
                matched_field = field
                break
        if matched_field is None or matched_field in results:
            continue

        qx0, qy0, qx1, qy1 = w["box"]
        candidates = []
        for j, w2 in enumerate(ocr_words):
            if j == i or j in used:
                continue
            vx0, vy0, vx1, vy1 = w2["box"]
            same_line = abs(((vy0 + vy1) / 2) - ((qy0 + qy1) / 2)) < (qy1 - qy0) * 1.6
            to_the_right = vx0 >= qx1 - 5
            if same_line and to_the_right:
                candidates.append((_distance(w["box"], w2["box"]), j, w2))

        candidates.sort(key=lambda t: t[0])
        chosen = candidates[:max_value_words]
        if chosen:
            value_tokens = [c[2]["text"] for c in chosen]
            confs = [c[2]["conf"] for c in chosen]
            for c in chosen:
                used.add(c[1])
            results[matched_field] = {
                "value": " ".join(value_tokens).strip().rstrip(".,;"),
                "word_confs": confs,
                "has_label": True,
            }
    return results


## 2. Validation sémantique : règles de format par champ

On vérifie que chaque valeur extraite respecte un format plausible pour son type de
champ. `DATE` est le seul champ avec un format strict et vérifiable automatiquement
(les autres — noms de personnes, sujets libres — n'ont pas de format contraignant).

In [ ]:
FORMAT_VALIDATORS = {
    "DATE": re.compile(
        r"(\d{1,2}[/\-.]\d{1,2}[/\-.]\d{2,4})|"
        r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s+\d{1,2},?\s+\d{2,4}",
        re.IGNORECASE,
    ),
}


def validate_format(field, value):
    """Retourne True si le champ n'a pas de contrainte connue, ou si le format est respecté."""
    if not value:
        return False
    if field not in FORMAT_VALIDATORS:
        return True
    return bool(FORMAT_VALIDATORS[field].search(value))


## 3. Score de confiance combiné

Combine 4 signaux pondérés, comme prévu au cahier des charges :
- **format** (35%) : conformité au format attendu
- **ocr_conf** (30%) : score de confiance OCR moyen des mots concernés
- **has_label** (20%) : présence d'un label détecté à proximité
- **stability** (15%) : stabilité de la valeur retrouvée sous plusieurs niveaux de
  dégradation (si les images dégradées de la Phase 2 sont disponibles ; sinon signal
  neutre).

In [ ]:
WEIGHTS = {"format": 0.35, "ocr_conf": 0.30, "has_label": 0.20, "stability": 0.15}


def ocr_confidence_score(word_confs):
    valid = [c for c in word_confs if c is not None and c >= 0]
    if not valid:
        return 0.5
    return max(0.0, min(1.0, sum(valid) / len(valid) / 100.0))


def stability_score(values):
    values = [v for v in values if v]
    if len(values) <= 1:
        return 0.5
    counts = Counter(values)
    _, freq = counts.most_common(1)[0]
    return freq / len(values)


def compute_confidence(field, value, word_confs=None, has_label=True, stability_values=None):
    word_confs = word_confs or []
    stability_values = stability_values or ([value] if value else [])

    scores = {
        "format": 1.0 if validate_format(field, value) else 0.0,
        "ocr_conf": ocr_confidence_score(word_confs),
        "has_label": 1.0 if has_label else 0.3,
        "stability": stability_score(stability_values),
    }
    total = sum(scores[k] * WEIGHTS[k] for k in WEIGHTS)
    return {"confidence": round(total, 3), "components": scores, "is_reliable": total >= 0.5}


## 4. Exécution : extraction + validation + score de confiance

> Sous-échantillon de 15 documents (cohérent avec la Phase 5), pour un temps
> d'exécution raisonnable.

In [ ]:
N_SAMPLE = 15
subset = manifest[:N_SAMPLE]

extraction_rows = []
confidence_records = []

for entry in subset:
    img_path = RAW_DIR / "images" / entry["image"]
    ann = json.loads((RAW_DIR / "annotations" / entry["annotation"]).read_text(encoding="utf-8"))
    gt = ground_truth_pairs(ann)

    img = cv2.imread(str(img_path))
    ocr_result = run_ocr_with_boxes(img)
    extracted = extract_layout_based(ocr_result["words"])

    all_fields = set(gt) | set(extracted)
    for field in sorted(all_fields):
        pred_info = extracted.get(field, {})
        pred_value = pred_info.get("value")
        gt_value = gt.get(field)

        conf_report = compute_confidence(
            field, pred_value,
            word_confs=pred_info.get("word_confs", []),
            has_label=pred_info.get("has_label", False),
        )

        is_correct = (
            pred_value is not None and gt_value is not None and
            (pred_value.strip().lower() == gt_value.strip().lower()
             or pred_value.strip().lower() in gt_value.strip().lower()
             or gt_value.strip().lower() in pred_value.strip().lower())
        )

        extraction_rows.append({
            "document": entry["image"], "field": field,
            "ground_truth": gt_value, "predicted": pred_value,
            "confidence": conf_report["confidence"], "is_reliable": conf_report["is_reliable"],
            "format_ok": conf_report["components"]["format"] == 1.0,
        })
        if pred_value is not None:
            confidence_records.append({"confidence": conf_report["confidence"], "is_correct": is_correct})

df_confidence = pd.DataFrame(extraction_rows)

results_dir = PROJECT_ROOT / "results" / "tables"
results_dir.mkdir(parents=True, exist_ok=True)
df_confidence.to_csv(results_dir / "phase6_confidence_scores.csv", index=False)
df_confidence.head(15)


## 5. Matrice d'erreurs détaillée

In [ ]:
def classify_match(pred, gt):
    gt_missing = gt is None or (isinstance(gt, float) and pd.isna(gt))
    pred_missing = pred is None or (isinstance(pred, float) and pd.isna(pred))
    if gt_missing:
        return "hors_perimetre" if not pred_missing else "n/a"
    if pred_missing:
        return "missing"
    p, g = str(pred).strip().lower(), str(gt).strip().lower()
    if p == g:
        return "exact_match"
    if p in g or g in p:
        return "partial_match"
    return "incorrect"


df_confidence["status"] = df_confidence.apply(
    lambda row: classify_match(row["predicted"], row["ground_truth"]), axis=1
)

error_dir = PROJECT_ROOT / "results" / "error_analysis"
error_dir.mkdir(parents=True, exist_ok=True)
df_confidence.to_json(error_dir / "phase6_error_matrix.json", orient="records", indent=2, force_ascii=False)

status_counts = df_confidence["status"].value_counts()
print("Répartition des statuts :")
print(status_counts)

pd.crosstab(df_confidence["status"], df_confidence["is_reliable"], margins=True)


## 6. Corrélation entre score de confiance et exactitude réelle

C'est le test le plus important de la Phase 6 : **le score de confiance est-il
réellement utile ?** S'il l'est, les extractions correctes devraient avoir des scores
plus élevés que les extractions incorrectes.

In [ ]:
df_corr = pd.DataFrame(confidence_records)

if not df_corr.empty and df_corr["is_correct"].nunique() > 1:
    correlation = df_corr["confidence"].corr(df_corr["is_correct"].astype(int))
else:
    correlation = None

high_conf = df_corr[df_corr["confidence"] >= 0.5]
low_conf = df_corr[df_corr["confidence"] < 0.5]

report = {
    "n_extractions_avec_valeur": len(df_corr),
    "correlation_confiance_exactitude": round(correlation, 3) if correlation is not None else None,
    "taux_bonnes_extractions_haute_confiance_pct": round(100 * high_conf["is_correct"].mean(), 1) if len(high_conf) else None,
    "taux_erreurs_detectees_basse_confiance_pct": round(100 * (1 - low_conf["is_correct"]).mean(), 1) if len(low_conf) else None,
}

with open(results_dir / "phase6_confidence_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

report


## 7. Analyse qualitative : exemples de champs fiables vs non fiables

In [ ]:
print("=== Exemples à confiance ÉLEVÉE ===")
display(df_confidence[df_confidence["is_reliable"]].sort_values("confidence", ascending=False).head(5)
        [["document", "field", "predicted", "ground_truth", "confidence", "status"]])

print("\n=== Exemples à confiance FAIBLE ===")
display(df_confidence[~df_confidence["is_reliable"]].sort_values("confidence").head(5)
        [["document", "field", "predicted", "ground_truth", "confidence", "status"]])


## 8. Résultats attendus de la Phase 6 — récapitulatif

- **Score de confiance par champ** : `results/tables/phase6_confidence_scores.csv`,
  combinant format, confiance OCR, présence de label, stabilité.
- **Matrice d'erreurs** : `results/error_analysis/phase6_error_matrix.json` — détail
  exact_match / partial_match / incorrect / missing par champ et document.
- **Rapport de corrélation** : `results/tables/phase6_confidence_report.json` —
  mesure si le score de confiance distingue effectivement les bonnes des mauvaises
  extractions.
- **Constat attendu** : compte tenu des scores F1 modestes obtenus en Phase 5
  (extraction difficile sur ce dataset), le score de confiance sert surtout à
  **flaguer les extractions à vérifier manuellement**, plutôt qu'à garantir un
  résultat fiable à 100% — cohérent avec la limite mentionnée dans le cahier des
  charges ("le score de confiance proposé restera une première approximation").
- **Limite à documenter** : échantillon réduit (15 documents) et champ `DATE` seul
  doté d'un validateur de format strict — les autres champs (TO, FROM, SUBJECT...)
  n'ont pas de contrainte de format vérifiable automatiquement, ce qui limite le
  signal "format" pour ces champs.
- **Fin du pipeline Phases 1-6** : le projet couvre désormais l'ensemble de la chaîne
  prévue par le cahier des charges — prétraitement → OCR → layout → extraction →
  validation → score de confiance.
